# Calculate Effects

In [119]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [145]:
run_local = True

if run_local:
    path_results = '../results/'
    path_data = '../data/clean_data/'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = '/content/drive/My Drive/Pol pub Proyecto - shared/Datos/'
    path_data = drive_path

In [128]:
# Leemos los datos
matched_grids = pd.read_parquet(os.path.join(path_data, 'matched_data.parquet'))
accidentes = pd.read_parquet(os.path.join(path_data, 'accidentes_ponderados_parquimetros.parquet'))

In [155]:
# Sample matched data
matched_data = (
    accidentes
    .merge(matched_grids, on='grid_id', how='inner')
    .assign(treatment=lambda df: np.where((df.type == 'treated') & (df.year >= 2019), 1, 0))
    # .drop(['total_accidentes'], axis=1)
    # .rename({'accidentes_ponderados':'total_accidentes'}, axis=1)
)

In [156]:
# Define outcome and treatment variables
Y = matched_data['total_accidentes']
T = matched_data['treatment']

# Add a constant for the regression intercept
X = sm.add_constant(T)

# Fit the regression model
model = sm.OLS(Y, X).fit()

# Print the summary of the regression model
# Guardamos el resultado
with open(os.path.join(path_results, 'total_accidentes_simple_ols.txt'), 'w') as f:
    f.write(model.summary().as_text())

In [157]:
# Create weights (1 for treated, 0.5 for control if each treated is matched with two controls)
weights = np.where(matched_data['treatment'] == 1, 1, 0.5)

# Fit the weighted regression model
weighted_model = sm.WLS(Y, X, weights=weights).fit()

# Print the summary of the weighted regression model
with open(os.path.join(path_results, 'total_accidentes_simple_wls.txt'), 'w') as f:
    f.write(weighted_model.summary().as_text())

Pasemos ahora a la redacción de la metología. Te voy a decir lo que hicimos y tu trabajo es darle el formato y redacción adecuada siguiendo el mismo estilo que hemos llevado hasta ahora. Recuerda que este texto es un artículo de investigación formal, por lo que esto se debe ver relfejado en el tipo de redacción que se utilice.

Para este esudio, se intentó hacer primero un approach de segmentación de las principales vialidades en tamaños de 500 metros, pues según estudios anteriores, este era el área de efecto de las cámaras de seguridad. Sin emargo, hacer estos cortes no fue posible, por lo que se tuvo que tomar un approach distinto. Se tienen la ubicación de las principales vialidades de Ciudad de México. Sobre esas se creó un grid de un tamaño que nosotros podemos especificar. Nuestra elección fue hacerlo de 250 por 250 cuadros, dando un total de 62,500 cuadrados mapeados sobre todas las vialidades. Se hacía una selección de aquellos que tuvieran al menos una vialidad sobre ellos y resultaba entonces que nuestras unidades de estudio eran un total de 8,332 celdas con al menos una vialidad dentro de ellas.

Para este estudio, las unidades estudiadas son precisamente esas celdas, que están sujetas a tener o no una cámara de seguridad en ellas (tratadas) y veíamos cómo esto afectaba el número de accidentes totales que ocurren en cada una de estas celdas (outcome). Al estar todas las unidades en coordenadas de latitud y longitud, era muy sencillo hacer el match para identificar tanto las vialidades como los accidnetes que ocurren en cada una.

Una vez creado el grid, el segundo paso consistía en asignar las variables perinentes a cada celda para poder crear el propensity score. Las variables que se le asignaron a cada celda fueron las siguientes (todas previas a 2019 que es cuando se hace el corte de tratamiento): número total de vialidades que están en la celda, número máximo de carriles, si tienen vialidades con dos sentidos, con un sentido, número total de accidnetes automovilísticos, atropellados, choques con lesionados, choques con prensados, choque sin lesionados, accidnetes que involucraron motocicletas, personas atrapadas o desbarrancadas, vehículos atrapados y número de volcaduras.

Cuando ya se tenían los features, se procedía a calcular el propensity score de cada una de ellas. Para esta parte usamos el código de Adrienne Kline del MIT quien tiene un paquete en python que se encarga de hacer todo esto. Se calcula el propensity score utilizando una regresión logística y especificamos que la muestra debía balancearse, pues en total teníamos tan solo 103 celdas con alguna cámara de velocidad (este número es menor a 140 pues había celdas con más de una cámara de seguridad, esto ocurre normalemnte porque hay dos muy pegadas, una en cada sentido de la vialidad). 

El sigiuente paso consistió en hacer el matching de las celdas. El mismo paquete de Adrienne nos ayudó a lograrlo, pues ya tiene funciones implementadas de Nearest Neighbor y de KNN. Según nuestras pruebas, el mejor balance se obtuvo cuando tomábamos KNN haciendo match de 1:2, es decir, para cada unidad tratada teníamos dos unidades no tratadas que tenían características muy similares.

Como todo buen modelo de propensity score matching, el siguiente paso era mostrar que nuestros dos grupos de tratamiento y de control pasaban las pruebas de balance. El paquete de Adrienne ayudó con esto, pues tiene una función que grafica precismente la métrica que necesitábamos: el standardized mean deviation (smd), pero no nos daba preubas de hipótesis para validar que efectivamente las distribuciones de las variables eran similares. Sin embargo, esto se logró haciendo inspección de las distribuciones, para ver el detalle de cada una de las variables ver anexos. Con esto ya teníamos confirmado que el match había sido exitoso y solo quedaba mostrar el efecto total.

Una vez ya teníamos nuestra muestra de control y de tratamiento, teníamos que encontrar el número de accidentes ponderados por la movilidad que había existido en los distintos años. Se sumaron el total de ingresos por parquímetros anuales y en cada celda se encontró el promedio de accidentes anuales (sin distinguir el tipo de accidente) para dividir estos promedios entre el total de ingresos (que había sido dividido entre 1,000,000 para no tener unidades tan grandes, pero al ser una constante no modificó la forma de la distribución de los ingresos). Al final terminamos con datos que tenían el siguiente formato: grid_id, total de accidnentes ponderados, año, tratamiento (booleano que era 0 si la celda no tenía cámara de velocidad o si el año era menor a 2019, y 1 si el año era 2019 o posterior y la celda tenía cámara de velocidad).

Para evaluar entonces el efecto se tomaron dos opciones: evaluar la diferencia de medias simple (ols) o de forma ponderada (wls) (porque para las celdas de control teníamos dos unidades por cada una de tratamiento). Se hicieron ambas y los resultados fueron muy similares, con tan solo pequeñas diferencias en la significancia del estimador del efecto. En los resutlados se muestran ambas.